In [2]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "APAI_colab_token"
os.environ["KAGGLE_KEY"] = "xx"

In [4]:
!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 13 file(s) under /src/
src/cl_strategies/naive.py -> /content/src/cl_strategies/naive.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/env_init.py -> /content/src/env_init.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/models/baselines.py -> /content/src/models/baselines.py
src/models/pretrained.py -> /content/src/models/pretrained.py
src/utils/save.py -> /content/src/utils/save.py

 All files saved under /content/src/
Contents of /content/src/:
/content/src/cl_strategies

In [5]:
!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh    # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

100% 6.53G/6.53G [01:00<00:00, 117MB/s] 
Extracting files...
Dataset path: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4

Detecting path and updating config.py...
Found in kagglehub cache: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
DATASET_ROOT = '/root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/'
OUTPUT_ROOT  = '/content/ucf101-processed/'

Setup complete!
config.py updated on Drive.
Restarting kernel in 5 seconds...
Dataset cache stays on disk.
Run Cell 3 after restart to verify.
Restarting in 5...
Restarting in 4...
Restarting in 3...
Restarting in 2...
Restarting in 1...


In [6]:
# imports
%run /content/src/imports.py

Using device: cuda


## **Continual Learning - Replay Buffer**

In [7]:
# Separate folders
EXEMPLAR_ROOT = "./UCF101/processed_exemplars"
SUBSET1_ROOT = "./UCF101/processed_subset1"

# PROCESS OLD CLASSES (The Memory Buffer)
# We limit 'train' to 20 samples per class to save space

limit = 20 
preprocess_dataset(
    target_classes=cfg.SELECTED_CLASSES, # The 10 original classes
    output_root=EXEMPLAR_ROOT,
    max_samples=limit
)

# PROCESS NEW CLASSES (Subset 1)
# We take 100% of the data for these new classes

preprocess_dataset(
    target_classes=cfg.TASK_1,
    output_root=SUBSET1_ROOT,
)


--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_exemplars
Classes: 10 | Limit: 20

Processing split: train



Processing split: val



Processing split: test



Done! Dataset ready at: ./UCF101/processed_exemplars

--- Preprocessing UCF101 ---
From: /root/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4/
To  : ./UCF101/processed_subset1
Classes: 10 | Limit: Full

Processing split: train



Processing split: val



Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset1


In [8]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1

# The "Master Map"
class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

print(class_to_idx)

{'PlayingTabla': 0, 'PommelHorse': 1, 'JumpingJack': 2, 'PushUps': 3, 'PoleVault': 4, 'HorseRace': 5, 'HighJump': 6, 'Drumming': 7, 'HorseRiding': 8, 'Diving': 9, 'ApplyEyeMakeup': 10, 'ApplyLipstick': 11, 'Archery': 12, 'BabyCrawling': 13, 'BalanceBeam': 14, 'BandMarching': 15, 'BlowDryHair': 16, 'BlowingCandles': 17, 'BodyWeightSquats': 18, 'Bowling': 19}


In [9]:
batch_size = 8

# Load the 20 exemplars per class we preprocessed earlier
old_exemplars_dataset = UCF101Clips(
    "./UCF101/processed_exemplars/train",
    class_to_idx=class_to_idx
)

val_base = UCF101Clips("./UCF101/processed_exemplars/val", class_to_idx=class_to_idx)
val_sub1 = UCF101Clips("./UCF101/processed_subset1/val", class_to_idx=class_to_idx) # Added missing definition for val_sub1

test_base = UCF101Clips("./UCF101/processed_exemplars/test", class_to_idx=class_to_idx)
test_sub1 = UCF101Clips("./UCF101/processed_subset1/test", class_to_idx=class_to_idx)

# Load the full 10 classes of Subset 1
new_classes_dataset = UCF101Clips(
    "./UCF101/processed_subset1/train",
    class_to_idx=class_to_idx
)

continual_train_dataset = ConcatDataset([old_exemplars_dataset, new_classes_dataset])

load_continual_train = torch.utils.data.DataLoader(
    continual_train_dataset,
    batch_size=batch_size, 
    shuffle=True,
    num_workers=2
)

load_combined_val_t1 = torch.utils.data.DataLoader(
    ConcatDataset([val_base, val_sub1]),
    batch_size=batch_size,
    shuffle=False
)

load_combined_test = torch.utils.data.DataLoader(
    ConcatDataset([test_base, test_sub1]),
    batch_size=batch_size,
    shuffle=False
)

In [11]:
import random

class ReplayBuffer:
    def __init__(self, max_size=1000):
        self.max_size = max_size
        self.data = []

    def add_batch(self, x, y):
        for i in range(len(x)):
            self.data.append((x[i].cpu(), y[i].cpu()))

        # keep buffer size fixed
        if len(self.data) > self.max_size:
            self.data = self.data[-self.max_size:]

    def sample(self, batch_size):
        if len(self.data) == 0:
            return None, None

        samples = random.sample(self.data, min(batch_size, len(self.data)))
        x, y = zip(*samples)
        return torch.stack(x), torch.tensor(y)

In [12]:
# Instantiate the replay buffer 
buffer = ReplayBuffer(max_size=500)

In [13]:
# Create a temporary loader just to fill the buffer
filler_loader = torch.utils.data.DataLoader(old_exemplars_dataset, batch_size=batch_size)

for x, y in filler_loader:
    # This fills the buffer with 20 samples per old class
    buffer.add_batch(x, y)

print(f"Buffer populated with {len(buffer.data)} old samples.")

Buffer populated with 200 old samples.


In [15]:
# Define teacher model
teacher = ResNet50LSTM().to(device)
params = torch.load(cfg.RESNET50_PATH, map_location=device)
teacher.load_state_dict(params)

<All keys matched successfully>